# Experimento C2: entorno estocástico con distinta demanda para c/ estación

### Carga de librerías

In [1]:
import random
import numpy as np
import matplotlib.pyplot as plt

import entorno
import agentes
import train

### Parámetros del entorno

In [2]:
TAMANOS = {
    'n_estaciones': 4,
    'anclas_por_est': 10,
    'bicis_por_est': 5,    # Estado inicial
    'bicis_por_accion': 2  # Bicis movidas por el agente en c/ accion
}
COSTOS = {
    'accion': 1,
    'accion_invalida': 10,
    'est_vacia': 1,
    'est_llena': 1,
}
TIEMPOS = {
    'viaje': 10,
    'accion': 5,
    'episodio': 8*60,
    'estado': 1,
    'update': 999
}
PROBS = {
    'origen': [0.1, 0.2, 0.3, 0.4],
    'destino': np.array(4*[4*[0.25]])
}


TRAIN_PARAMS = {
    'alpha_init': 0.1,
    'alpha_step': 1,
    'alpha_end':  0.1,

    'eps_init': 0.7,
    'eps_step': 1,
    'eps_end':  0.7
}

### Entrenar agentes

In [3]:
random.seed(10)    # Aleatoriedad del agente
np.random.seed(10) # Aleatoriedad del entorno

env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)

agente_Q = agentes.QLearningAgent(action_size=env.action_space.n, alpha=TRAIN_PARAMS['alpha_init'], epsilon=TRAIN_PARAMS['eps_init'])
_, metricas_train_test, counters_train_test, best_qtable = train.train_test(env, agente_Q, params=TRAIN_PARAMS, N=10**5, keep_best=True, verbose=False)

Entrenamiento finalizado en 22.8 minutos.


In [4]:
random.seed(777)    # Aleatoriedad del agente
np.random.seed(777) # Aleatoriedad del entorno

env = entorno.Entorno(TAMANOS, COSTOS, TIEMPOS, PROBS)

print("=== Agente Nulo ===")
agente_N = agentes.NullAgent()
_, _, _, _ = train.train(env, agente_N, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Aleatorio ===")
agente_R = agentes.RandomAgent(action_size=env.action_space.n)
_, _, _, _ = train.train(env, agente_R, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Heurístico v1 ===")
agente_H = agentes.HeuristicAgent(action_size=env.action_space.n)
_, _, _, _ = train.train(env, agente_H, params=TRAIN_PARAMS, N=10**3)

print("=== Agente Q-learning ===")
_, metricas_test, counters_test, _ = train.test(env, agente_Q.q_table, N=10**3)

print("=== Agente Heurístico v2 ===")
agente_H2 = agentes.HeuristicAgent2(action_size=env.action_space.n)
_, _, counters_h2, _ = train.train(env, agente_H2, params=TRAIN_PARAMS, N=10**3)

=== Agente Nulo ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 174.0 ± 18.2 (36.3 ± 3.4%)
# Prolongados: 30.7 ± 10.9 (10.1 ± 3.7%)
Tiempo desbalanceo: 801.4 ± 78.9 (41.7 ± 4.1%)
Recompensa: -204.7 ± 22.3
=== Agente Aleatorio ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 141.5 ± 18.4 (29.4 ± 3.5%)
# Prolongados: 13.6 ± 7.3 (4.1 ± 2.3%)
Tiempo desbalanceo: 598.3 ± 70.7 (31.2 ± 3.7%)
Recompensa: -714.6 ± 113.4
=== Agente Heurístico v1 ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 116.8 ± 14.8 (24.3 ± 2.7%)
# Prolongados: 0.0 ± 0.2 (0.0 ± 0.1%)
Tiempo desbalanceo: 428.4 ± 35.5 (22.3 ± 1.8%)
Recompensa: -140.5 ± 16.1
=== Agente Q-learning ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 60.5 ± 10.8 (12.5 ± 2.1%)
# Prolongados: 1.4 ± 1.7 (0.3 ± 0.4%)
Tiempo desbalanceo: 366.9 ± 40.4 (19.1 ± 2.1%)
Recompensa: -116.5 ± 14.2
=== Agente Heurístico v2 ===
Entrenamiento finalizado en 0.1 minutos.
# Insatisfechos: 97.4 ± 12.5 (20.2 ± 2.2%)

In [5]:
decisiones = {k: np.argmax(v) for k,v in agente_Q.q_table.items()}
conteo_decisiones = dict()
for i in range(13):
    conteo_decisiones[i] = len([x for x in decisiones.values() if x == i])
conteo_decisiones

{0: 4187,
 1: 281,
 2: 505,
 3: 1056,
 4: 110,
 5: 265,
 6: 436,
 7: 82,
 8: 107,
 9: 147,
 10: 48,
 11: 67,
 12: 106}

In [6]:
#counters_test['acciones']
dict(sorted(dict(counters_test['acciones']).items()))

{0: 166856,
 1: 736,
 2: 7752,
 3: 26078,
 4: 205,
 5: 3921,
 6: 10182,
 7: 137,
 8: 290,
 9: 2563,
 10: 93,
 11: 242,
 12: 378}

In [7]:
sum(counters_test['acciones'].values())

219433

In [8]:
dict(sorted(dict(counters_h2['acciones']).items()))

{0: 299751,
 1: 3772,
 2: 8768,
 3: 10681,
 4: 97,
 5: 2720,
 6: 3071,
 7: 26,
 8: 272,
 9: 654,
 10: 4,
 11: 63,
 12: 90}

In [9]:
sum(counters_h2['acciones'].values())

329969